# ColBERT (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
ColBERT = Contextualized Late Interaction over BERT

ColBERT — это инновационный подход к Dense Retrieval, разработанный исследователями из Стэнфордского университета. Он позволяет достичь высокой точности, характерной для моделей с поздней интеракцией (cross-encoders), сохраняя при этом эффективность масштабирования, присущую моделям с ранней интеракцией (two-tower models).

### Контекст

Задача информационного поиска в крупных корпусах документов традиционно балансирует между точностью и скоростью. До появления ColBERT существовало два основных лагеря нейронных методов:

1.  **Dense Retrieval (ранняя интеракция)**: Такие модели, как DPR (2020), используют две отдельные нейронные сети (обычно BERT-подобные энкодеры) для кодирования запроса и документа в единые фиксированные плотные векторы (embeddings). Релевантность определяется простым скалярным произведением (dot product) этих векторов.
    *   **Преимущества**: Высокая масштабируемость. Векторы документов можно предварительно вычислить и сохранить в ANN-индексе (Approximate Nearest Neighbor), что делает поиск очень быстрым.
    *   **Недостатки**: Ограниченная выразительность. "Ранняя интеракция" означает, что весь запрос и весь документ сжимаются в один вектор *до* какой-либо интеракции между ними. Это создает **семантический бутылочное горлышко (semantic bottleneck)**, теряя тонкие, токен-уровневые взаимодействия, которые могут быть критически важны для определения релевантности.

2.  **Cross-Encoders (поздняя интеракция)**: Эти модели (например, BERT, используемый как reranker (2018)) конкатенируют запрос и документ, а затем подают их оба в один большой Transformer. Модель напрямую выдает оценку релевантности, учитывая все возможные взаимодействия между токенами запроса и документа.
    *   **Преимущества**: Высочайшая точность, поскольку модель может изучать сложные, тонкие зависимости между словами запроса и документа.
    *   **Недостатки**: Крайне неэффективно для первоначального поиска по большому корпусу. Для каждого запроса и *каждого* документа в корпусе (или даже тысяч релевантных документов) требуется полный проход через Transformer, что делает этот подход непригодным для масштабирования за пределами reranking уже отобранных документов.

### Идея метода

ColBERT предлагает преодолеть компромисс между точностью и скоростью, объединив лучшее из обоих миров: **высокую выразительность поздней интеракции** с **эффективностью индексации ранней интеракции**.

Основная идея ColBERT заключается в следующем:

1.  **Многовекторное представление**: Вместо того чтобы представлять запрос или документ одним плотным вектором, ColBERT представляет каждый из них как **набор (или матрицу) плотных векторов**, где каждый вектор соответствует отдельному токену (или его фрагменту). Эти векторы *контекстуализированы* с помощью BERT, то есть каждый токен кодируется с учетом окружающих его слов.
2.  **Эффективная "поздняя" интеракция**: При вычислении релевантности ColBERT не выполняет полную Transformer-интеракцию, как cross-encoders. Вместо этого он использует специализированную, но эффективную операцию **MaxSim (Maximum Similarity)**, которая вычисляет агрегированную максимальную попарную схожесть между векторами токенов запроса и векторами токенов документа.
3.  **Предварительная индексация**: Поскольку документ кодируется независимо от запроса (как набор векторов), эти наборы векторов могут быть предварительно вычислены и проиндексированы. Это позволяет ColBERT использовать ANN-индексы, подобно Dense Retrieval моделям, для быстрого поиска.

Таким образом, ColBERT позволяет модели "заглянуть" внутрь документа и запроса на уровне токенов, имитируя тонкие взаимодействия cross-encoders, но при этом сохраняет возможность предварительного индексирования, необходимого для масштабируемости.

### Постановка задачи

Решается задача **Dense Retrieval**: по заданному запросу $Q$ и корпусу документов $D = \{d_1, d_2, ..., d_N\}$, необходимо найти $K$ наиболее релевантных документов. Релевантность определяется таким образом, чтобы помочь в решении конечных задач, таких как Question Answering или обычный поиск информации.

### Альтернативные методы на момент появления ColBERT (2020)

1.  **Sparse Retrieval (например, BM25)**:
    *   **Механизм**: Использует статистические методы на основе частоты слов и обратной частоты документов. Интеракция происходит на уровне совпадения ключевых слов.
    *   **Отличие**: Не использует нейронные сети и не улавливает семантическое значение слов или фраз, что приводит к проблемам с синонимами и полисемией.
2.  **Dense Retrieval (например, DPR (2020))**:
    *   **Механизм**: Двухбашенные модели (two-tower models), где запрос и документ кодируются отдельными BERT-энкодерами в один плотный вектор каждый. Схожесть вычисляется скалярным произведением.
    *   **Отличие**: ColBERT также использует BERT для кодирования, но *не сжимает* запрос и документ в один вектор. Вместо этого, ColBERT сохраняет множественные, токен-уровневые векторы, что позволяет более богатую интеракцию. DPR страдает от семантического бутылочного горлышка.
3.  **Cross-Encoders (например, BERT-based Rerankers (2018))**:
    *   **Механизм**: Конкатенация запроса и документа, проход через один BERT-энкодер для получения оценки релевантности.
    *   **Отличие**: ColBERT стремится к точности cross-encoders, но без их вычислительных затрат на индексацию. ColBERT *предварительно вычисляет* представления документов, тогда как cross-encoders должны обрабатывать каждую пару запрос-документ "на лету".

### Архитектура модели

ColBERT использует **один общий BERT-энкодер** для кодирования как запросов, так и документов. Однако способ получения и использования выходных представлений отличается:

1.  **Кванторизатор токенов (Tokenizer)**: Стандартный WordPiece или BPE токенизатор для разбиения текста на токены.
2.  **BERT Энкодер**: Основной компонент. Принимает последовательность токенов и выдает контекстуализированные векторные представления для *каждого* токена.
    *   **Для запросов**: Запрос $Q$ форматируется как `[CLS] Q [SEP]`. Затем он пропускается через BERT. Из выходных представлений **отбрасывается выход `[CLS]`** (который обычно используется для классификации всего предложения). Вместо этого, для каждого *значимого токена* запроса (т.е. всех токенов, кроме `[CLS]`, `[SEP]`, и иногда `[MASK]`) извлекается его соответствующий выходной вектор. Это формирует матрицу векторов $E_Q = \{e_{q_1}, e_{q_2}, ..., e_{q_m}\}$, где $m$ — количество токенов в запросе.
    *   **Для документов**: Документ $D$ форматируется аналогично как `[CLS] D [SEP]`. Он также пропускается через тот же BERT. Идентично запросам, **отбрасывается выход `[CLS]`**. Для каждого *значимого токена* документа извлекается его выходной вектор, формируя матрицу $E_D = \{e_{d_1}, e_{d_2}, ..., e_{d_k}\}$, где $k$ — количество токенов в документе.

Ключевое отличие от Dense Retrieval: ColBERT не использует *pooling* операцию (например, усреднение или использование `[CLS]`-токена) для сжатия всего запроса или документа в один вектор. Вместо этого, сохраняются *все* контекстуализированные векторные представления токенов.

### Алгоритм обучения

Обучение ColBERT базируется на **контрастном обучении (contrastive learning)**, схожем с DPR. Цель — научить модель давать высокие оценки релевантности для положительных пар (запрос, релевантный документ) и низкие для отрицательных пар (запрос, нерелевантный документ).

1.  **Входные данные**: На каждом шаге обучения модель получает:
    *   Запрос $Q$.
    *   Один положительный документ $D^+$ (действительно релевантный запросу).
    *   Несколько отрицательных документов $D^-_i$. Авторы ColBERT, как и DPR, подчеркивают важность использования **hard negatives** (трудных отрицательных примеров), которые похожи на запрос, но нерелевантны. Эти hard negatives могут быть получены, например, с помощью BM25 или других Dense Retrieval моделей, или путем майнинга из батча.
2.  **Кодирование**:
    *   Все токены запроса $Q$, положительного документа $D^+$, и всех отрицательных документов $D^-_i$ пропускаются через общий BERT-энкодер для получения их контекстуализированных токен-векторов.
    *   $E_Q = \text{BERT}(Q)$
    *   $E_{D^+} = \text{BERT}(D^+)$
    *   $E_{D^-_i} = \text{BERT}(D^-_i)$
    *   Важно: `[CLS]`-токены игнорируются. Для всех токен-векторов применяется L2-нормализация.
3.  **Вычисление оценки релевантности (MaxSim)**:
    *   Для каждой пары (запрос, документ) вычисляется оценка релевантности с использованием оператора **MaxSim**:
        $Score(Q, D) = \sum_{e_q \in E_Q} \max_{e_d \in E_D} (e_q \cdot e_d)$
    *   Где $e_q$ — это токен-вектор из запроса, а $e_d$ — токен-вектор из документа. Операция `max` ищет для каждого токена запроса наиболее похожий токен в документе (используя скалярное произведение, поскольку векторы L2-нормализованы, это эквивалентно косинусной близости). Затем эти максимальные схожести суммируются по всем токенам запроса. Это позволяет захватить "позднюю интеракцию", поскольку каждый токен запроса "взаимодействует" с каждым токеном документа.
4.  **Функция потерь (Loss Function)**:
    *   Используется стандартная **кросс-энтропийная функция потерь (Cross-Entropy Loss)**, также известная как отрицательное логарифмическое правдоподобие (Negative Log-Likelihood) поверх Softmax:
        $\mathcal{L} = -\log \frac{\exp(Score(Q, D^+))}{\sum_{D' \in \{D^+\} \cup \{D^-\}} \exp(Score(Q, D'))}$
    *   Эта функция потерь максимизирует вероятность того, что положительный документ $D^+$ будет иметь наивысший балл релевантности среди всех рассматриваемых документов (положительного и отрицательных).

### Алгоритм инференса (поиска)

1.  **Индексация документов (Pre-computation)**:
    *   Перед началом поиска, все документы в корпусе должны быть проиндексированы.
    *   Для каждого документа $d_j$:
        *   Он пропускается через обученный ColBERT BERT-энкодер для получения его токен-векторов $E_{d_j}$.
        *   Эти наборы векторов $E_{d_j}$ (матрицы) сохраняются в специализированном индексе. ColBERT часто использует **FAISS** (Facebook AI Similarity Search) или его модификации, адаптированные для эффективного поиска по MaxSim.
    *   Это ключевой шаг, который делает ColBERT масштабируемым: представления документов вычисляются *один раз* и сохраняются.
2.  **Обработка запроса (Query Processing)**:
    *   Когда приходит новый запрос $Q$:
        *   Он пропускается через тот же обученный ColBERT BERT-энкодер для получения его токен-векторов $E_Q$.
        *   Эти векторы готовы для интеракции с проиндексированными документами.
3.  **Поиск (Retrieval)**:
    *   Используя $E_Q$, система ищет в предварительно построенном индексе документов.
    *   Для каждого потенциально релевантного документа $d_j$ из индекса (или для всех документов, если индекс не используется):
        *   Вычисляется $Score(Q, d_j)$ с использованием оператора MaxSim между $E_Q$ и $E_{d_j}$.
        *   Оптимизация: В реальных системах, для ускорения, MaxSim вычисляется не со всеми документами, а с помощью приближенных методов, интегрированных с ANN-индексами, чтобы эффективно находить документы с высокими MaxSim-оценками.
    *   Возвращаются $K$ документов с наивысшими оценками релевантности.

### Результаты

ColBERT был протестирован на нескольких стандартных бенчмарках для информационного поиска, таких как **MS MARCO Passage Ranking** и **TREC Car**.

Основные выводы из сравнения с другими методами:

*   **Значительное улучшение точности по сравнению с Dense Retrieval**: На наборе MS MARCO Passage Ranking, ColBERT показал прирост метрики **MRR@10 (Mean Reciprocal Rank at 10)** примерно на **10-15 процентных пунктов** по сравнению с DPR и другими одно-векторными Dense Retrieval моделями. Например, ColBERT достиг MRR@10 около 35%, тогда как DPR — около 25%. Это демонстрирует эффективность поздней интеракции на токен-уровне.
*   **Сравнительная или превосходящая точность по сравнению с Cross-Encoders**: В задачах reranking, где cross-encoders традиционно доминируют, ColBERT часто достигал сравнимой точности. А в сценариях, где требовался прямой (first-stage) retrieval, ColBERT показывал значительно лучшую точность, чем Dense Retrieval, приближаясь к верхней границе, установленной cross-encoders, но при этом будучи масштабируемым.
*   **Ускорение поиска**: При выполнении первого этапа поиска (first-stage retrieval), ColBERT был **в сотни раз быстрее**, чем полноценные cross-encoders, сохраняя при этом их высокую точность. По сравнению с Dense Retrieval, ColBERT может быть немного медленнее (из-за более сложной операции MaxSim и хранения большего количества векторов), но предлагает значительно лучшую точность при приемлемых накладных расходах.
*   **Эффективное использование памяти**: Хотя ColBERT хранит больше векторов на документ, чем Dense Retrieval, общая память, необходимая для индекса, оказывается управляемой, особенно с учетом техник квантизации векторов, используемых в ColBERTv2.

В целом, ColBERT успешно **заполнил пробел между скоростью и точностью** в нейронном информационном поиске, предложив модель, которая способна выполнять эффективный, полномасштабный поиск с качеством, близким к лучшим rerankers.

## 📝 Критический анализ

```markdown
# ColBERT (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
ColBERT = Contextualized Late Interaction over BERT

ColBERT — это подход к Dense Retrieval, разработанный в Стэнфорде. Он сочетает точность cross-encoders с масштабируемостью two-tower models.

### Контекст

Информационный поиск балансирует между точностью и скоростью. До ColBERT существовали:

1. **Dense Retrieval**: Использует два энкодера для кодирования запроса и документа в векторы. Преимущество — масштабируемость, недостаток — семантическое бутылочное горлышко.
2. **Cross-Encoders**: Конкатенируют запрос и документ для оценки релевантности. Преимущество — точность, недостаток — низкая эффективность.

### Идея метода

ColBERT объединяет точность и эффективность:

1. **Многовекторное представление**: Запрос и документ представлены как набор векторов, каждый из которых соответствует токену.
2. **Эффективная "поздняя" интеракция**: Использует MaxSim для вычисления схожести между токенами.
3. **Предварительная индексация**: Векторы документов предварительно вычисляются и индексируются.

### Постановка задачи

Решается задача **Dense Retrieval**: по запросу $Q$ и корпусу документов $D$ найти $K$ наиболее релевантных документов.

### Альтернативные методы

1. **Sparse Retrieval (BM25)**: Использует частоту слов, не улавливает семантику.
2. **Dense Retrieval (DPR)**: Двухбашенные модели, страдают от семантического бутылочного горлышка.
3. **Cross-Encoders**: Высокая точность, но низкая эффективность.

### Архитектура

ColBERT использует **BERT-энкодер** для кодирования запросов и документов. Запросы и документы форматируются как `[CLS] Q [SEP]` и `[CLS] D [SEP]`, соответственно. Извлекаются векторы токенов, формируя матрицы $E_Q$ и $E_D$.

### Алгоритм обучения

Обучение базируется на **контрастном обучении**:

1. **Входные данные**: Запрос $Q$, положительный документ $D^+$, отрицательные документы $D^-_i$.
2. **Кодирование**: Все токены кодируются BERT-энкодером.
3. **MaxSim**: $Score(Q, D) = \sum_{e_q \in E_Q} \max_{e_d \in E_D} (e_q \cdot e_d)$.
4. **Функция потерь**: Кросс-энтропийная функция потерь.

### Алгоритм инференса

1. **Индексация документов**: Документы кодируются и индексируются.
2. **Обработка запроса**: Запрос кодируется.
3. **Поиск**: Используется MaxSim для поиска релевантных документов.

### Результаты

ColBERT показал улучшение на **10-15 п.п.** в MRR@10 на MS MARCO по сравнению с DPR. Он быстрее cross-encoders и обеспечивает высокую точность при приемлемых затратах на память.

<img src="img/img.png" width=500>

ColBERT успешно сочетает скорость и точность в нейронном информационном поиске.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций ColBERT на Python
# Используем библиотеки Hugging Face Transformers и FAISS для иллюстрации

from transformers import BertTokenizer, BertModel
import torch
import faiss
import numpy as np

# Инициализация токенизатора и модели BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Функция для кодирования текста в токен-векторы с использованием BERT
def encode_text(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    # Отбрасываем [CLS] и [SEP] токены, берем только значимые токены
    token_embeddings = outputs.last_hidden_state[:, 1:-1, :]
    # L2-нормализация токен-векторов
    token_embeddings = torch.nn.functional.normalize(token_embeddings, p=2, dim=2)
    return token_embeddings.squeeze(0).detach().numpy()

# Пример документов и запросов
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast brown fox leaps over a sleepy dog.",
    "The quick red fox jumps over the lazy cat."
]

query = "A quick fox jumps over a lazy dog."

# Кодирование документов и запроса
document_embeddings = [encode_text(doc) for doc in documents]
query_embedding = encode_text(query)

# Индексация документов с использованием FAISS
dimension = document_embeddings[0].shape[1]
index = faiss.IndexFlatIP(dimension)  # Используем Inner Product (скалярное произведение)
for doc_embedding in document_embeddings:
    index.add(doc_embedding)

# Функция для вычисления MaxSim между запросом и документом
def maxsim(query_embedding, document_embedding):
    # Для каждого токена запроса находим максимальную схожесть с токенами документа
    max_similarities = np.max(np.dot(query_embedding, document_embedding.T), axis=1)
    # Суммируем максимальные схожести
    return np.sum(max_similarities)

# Поиск наиболее релевантного документа
scores = [maxsim(query_embedding, doc_embedding) for doc_embedding in document_embeddings]
best_doc_index = np.argmax(scores)

print(f"Most relevant document: {documents[best_doc_index]}")

# Вывод: "Most relevant document: The quick brown fox jumps over the lazy dog."
```

### Комментарии к коду:

1. **Токенизация и кодирование**: Используем `BertTokenizer` и `BertModel` из библиотеки Hugging Face для получения контекстуализированных токен-векторов. Мы отбрасываем `[CLS]` и `[SEP]` токены, как это делается в ColBERT, и нормализуем векторы.

2. **Индексация с FAISS**: FAISS используется для индексации токен-векторов документов. Это позволяет быстро находить документы с максимальной схожестью.

3. **MaxSim**: Реализуем операцию MaxSim, которая для каждого токена запроса находит максимальную схожесть с токенами документа и суммирует эти максимальные значения.

4. **Поиск**: Вычисляем MaxSim для каждого документа и выбираем документ с наивысшей оценкой как наиболее релевантный.

Этот пример иллюстрирует, как ColBERT использует многовекторное представление и позднюю интеракцию для достижения высокой точности в задачах информационного поиска.